# DATA1002 Project Stage 1

**Author:** 550085605 - Tuan Phuc Tran

**Topic:** Timeliness of NSW public emergency departments, 2010–2026  

**Data source:** Bureau of Health Information (BHI)

**Individual question:** How does the timeliness of starting emergency treatment in NSW differ across triage categories and Local Health Districts, and has the gap widened over time?


In [ ]:
#Import libraries
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

## 1. Data Loading

The BHI workbook contains six sheets. Two are used in this analysis:

- **Results by triage** — treatment timeliness split by triage category
- **Overall results** — attendance counts, used later to compare workload

The other four sheets cover measures outside the scope of this question.

In [ ]:
#Loading data 
# Path.cwd() is the notebook's own folder, so the code runs unchanged elsewhere
RAW_FILE = Path.cwd() / "BHI_DATA_HQ_ED_Jan2010Mar2026.xlsx"

triage_raw = pd.read_excel(RAW_FILE, sheet_name="Results by triage")
overall_raw = pd.read_excel(RAW_FILE, sheet_name="Overall results")

print(f"Triage : {triage_raw.shape[0]:,} rows x {triage_raw.shape[1]} columns")
print(f"Overall: {overall_raw.shape[0]:,} rows x {overall_raw.shape[1]} columns")
triage_raw.head(3)

### Dataset structure

The data is in **long format**: each row records one measure, for one entity, in one quarter. Four columns describe what a row contains such as `measure_name`, `subcategory`, `reporting_level` and `reporting_period` which are while the actual number sits in `measure_value`.

In [ ]:
# One row = one measure, for one entity, in one quarter.
# These four columns define what each row holds.
for col in ["measure_name", "subcategory", "reporting_level"]:
    print(f"\n{col}:")
    print(triage_raw[col].value_counts().to_string())

print(f"\nreporting_period: {triage_raw['reporting_period'].nunique()} quarters, "
      f"{triage_raw['reporting_period'].iloc[0]} to {triage_raw['reporting_period'].iloc[-1]}")

### Initial observations
108,664 rows spanning 65 quarters (Jan–Mar 2010 to Jan–Mar 2026), across four reporting levels: NSW as a whole, seven peer groups, seventeen Local Health Districts, and individual hospitals.
Two features shape everything that follows. The levels are **stacked in one table**, so an analysis must filter to a single level or the same patients get counted three times over. And `measure_value` **mixes percentages, minutes and counts** in one column, so measures cannot be compared until one is selected.

## 2. Data quality assessment
Checks run on the raw data before any cleaning. Nothing is modified here, the figures below are the evidence cited in Section B of the report.

In [ ]:
def section(title):
    """Print a heading so each check is easy to find in the output."""
    print(f"\n{title}\n" + "-" * len(title))


section("MISSING VALUES")
# lhd_shortname and peer_group_shortname are blank by design: a peer group has
# no single LHD, and an LHD spans several peer groups. Only measure_value matters.
blank = triage_raw["measure_value"].isna().sum()
print(f"Blank measure_value: {blank:,} of {len(triage_raw):,} rows ({100*blank/len(triage_raw):.2f}%)\n")
display(triage_raw["note_text"].value_counts().to_frame("rows"))

section("DUPLICATES")
# A result is uniquely identified by these five columns together
row_id = ["reporting_period", "reporting_level", "entity_id", "measure_name", "subcategory"]
print("Identical rows  :", triage_raw.duplicated().sum())
print("Repeated records:", triage_raw.duplicated(subset=row_id).sum())

section("INCONSISTENT VALUES")
# IDs and names should match one to one; different counts would mean the same
# hospital appears under two spellings.
print("Entity IDs:", triage_raw["entity_id"].nunique(),
      "| Entity names:", triage_raw["entity_shortname"].nunique())
display(triage_raw["metric"].value_counts().to_frame("rows"))
display(triage_raw.groupby("subcategory")["measure_name"].nunique().to_frame("measures reported"))

In [ ]:
section("OUTLIERS")
# Hospital level only: NSW, LHD and peer group rows aggregate these same
# patients, so mixing levels would distort the distribution.
on_time = triage_raw[
    (triage_raw["measure_name"] == "Percentage of patients starting treatment on time")
    & (triage_raw["reporting_level"] == "Hospital")
]
display(on_time["measure_value"].describe().round(1).to_frame("on-time % across hospitals"))

section("FORMATTING")
# reporting_period is text, so alphabetical sorting puts April before January
print("reporting_period type:", triage_raw["reporting_period"].dtype)
print("Sorted as text:", sorted(triage_raw["reporting_period"].unique())[:3])

### Quality issues found

| Issue | Finding | Treatment |
|---|---|---|
| Missing values | 1,324 blanks (1.22%). BHI explains 724 as suppressed for small numbers or privacy; **600 have no reason given** | Removed, not imputed |
| Duplicates | None found, on either check | No action needed |
| Inconsistent values | 104 IDs map to 104 names | No action needed |
| Mixed units | `measure_value` holds percentages, minutes and counts together | One measure selected before analysis |
| Coverage gap | T1 Resuscitation reports patient counts only, no timeliness result | Analysis limited to T2–T5 |
| Outliers | Hospital results span 10.6% to 100.0% (median 77.4%) | Retained — extremes are real performance, not errors |
| Formatting | `reporting_period` is text, so it sorts alphabetically | Converted to a date in cleaning |

The 600 unexplained blanks and the T1 gap are limitations of the source data rather than problems cleaning can fix. Both carry into Section B.

## 3. Cleaning

Six decisions turn the raw long-format table into two analysis-ready tables. Each is justified in Section B of the report.

In [ ]:
# The single measure this project analyses. Chosen because it is the only
# quality measure reported across all 65 quarters; the four-hour measure was
# discontinued after Oct-Dec 2024 when NSW Health changed its access targets.
MEASURE = "Percentage of patients starting treatment on time"

section("FILTERING")
triage = triage_raw[triage_raw["measure_name"] == MEASURE].copy()
print(f"Rows for the chosen measure: {len(triage):,}")

# T1 Resuscitation is excluded: BHI reports patient counts for T1 but no
# timeliness result, since those patients are treated immediately.
triage = triage[triage["subcategory"] != "T1: Resuscitation"]
print(f"After removing T1        : {len(triage):,}  (T1 had no rows for this measure)")

# Suppressed results are removed rather than estimated. Filling them in would
# invent data that BHI deliberately withheld to protect privacy.
before = len(triage)
triage = triage.dropna(subset=["measure_value"])
print(f"After removing suppressed: {len(triage):,}  ({before - len(triage)} removed)")

In [ ]:
section("TRANSFORMING")

# reporting_period is text, so it sorts alphabetically. Mapping each quarter to
# its first month gives a real date that sorts and plots correctly.
QUARTER_START = {"Jan-Mar": 1, "Apr-Jun": 4, "Jul-Sep": 7, "Oct-Dec": 10}

def quarter_to_date(period):
    quarter, year = period.rsplit(" ", 1)
    return pd.Timestamp(int(year), QUARTER_START[quarter], 1)

triage["quarter_start"] = triage["reporting_period"].apply(quarter_to_date)
print("Time range:", triage["quarter_start"].min().date(),
      "to", triage["quarter_start"].max().date())

# NSW Health's own classification of its districts. Added so the analysis can
# test whether a city-country divide explains the differences between districts.
METRO = ["Central Coast", "Illawarra Shoalhaven", "Nepean Blue Mountains",
         "Northern Sydney", "South Eastern Sydney", "South Western Sydney",
         "Sydney", "Western Sydney"]
REGIONAL = ["Far West", "Hunter New England", "Mid North Coast",
            "Murrumbidgee", "Northern NSW", "Southern NSW", "Western NSW"]

def classify_district(name):
    if name in METRO:
        return "Metropolitan"
    if name in REGIONAL:
        return "Regional"
    # St Vincent's and Sydney Children's are statewide networks, not districts
    return "Specialty network"

triage["district_type"] = triage["lhd_shortname"].apply(classify_district)
display(triage.groupby("district_type")["lhd_shortname"].nunique().to_frame("districts"))

In [ ]:
section("TIDYING")

# Shorter names make the analysis code below much easier to read
triage_clean = triage[[
    "quarter_start", "reporting_period", "reporting_level", "entity_shortname",
    "lhd_shortname", "district_type", "peer_group_shortname",
    "subcategory", "measure_value"
]].rename(columns={
    "entity_shortname": "entity",
    "lhd_shortname": "lhd",
    "peer_group_shortname": "peer_group",
    "subcategory": "triage_category",
    "measure_value": "pct_on_time"
}).sort_values(["quarter_start", "reporting_level", "entity", "triage_category"])

print(f"Cleaned triage table: {len(triage_clean):,} rows")
display(triage_clean.head(3))

In [ ]:
section("HOSPITAL TABLE")

# Attendances and on-time results sit in separate rows, so the table is reshaped
# to put them side by side and let workload be compared against performance.
hospitals = overall_raw[
    (overall_raw["reporting_level"] == "Hospital")
    & (overall_raw["measure_name"].isin([MEASURE, "Attendances"]))
]

hospitals_clean = hospitals.pivot_table(
    index=["reporting_period", "entity_shortname", "lhd_shortname", "peer_group_shortname"],
    columns="measure_name",
    values="measure_value",
    aggfunc="first"
).reset_index()

hospitals_clean.columns.name = None    # drop the leftover pivot label
hospitals_clean = hospitals_clean.rename(columns={
    MEASURE: "pct_on_time",
    "Attendances": "attendances",
    "entity_shortname": "entity",
    "lhd_shortname": "lhd",
    "peer_group_shortname": "peer_group"
})

# A hospital is only usable here if both numbers are present
before = len(hospitals_clean)
hospitals_clean = hospitals_clean.dropna(subset=["pct_on_time", "attendances"])
hospitals_clean["quarter_start"] = hospitals_clean["reporting_period"].apply(quarter_to_date)
hospitals_clean["district_type"] = hospitals_clean["lhd"].apply(classify_district)

print(f"Cleaned hospital table: {len(hospitals_clean):,} rows  ({before - len(hospitals_clean)} incomplete pairs removed)")
display(hospitals_clean.head(3))

In [ ]:
#Saving clean data file
# Saved beside the notebook so the marker can open them without changing paths.
triage_clean.to_csv(Path.cwd() / "ed_triage_clean.csv", index=False)
hospitals_clean.to_csv(Path.cwd() / "ed_hospitals_clean.csv", index=False)

### Cleaning decisions

| Decision | Reason |
|---|---|
| Analyse "% starting treatment on time" | The only quality measure reported across all 65 quarters; the four-hour measure ends at Oct–Dec 2024 |
| Exclude T1 Resuscitation | BHI reports no timeliness result for T1 — the filter removed 0 rows, confirming this |
| Remove 144 suppressed values | Imputing them would invent data BHI withheld for privacy |
| Convert quarters to dates | Text quarters sort alphabetically, putting April before January |
| Add `district_type` | NSW Health's metropolitan/regional split, used to test the city-country divide |
| Keep two separate tables | Different units of observation; merging would repeat attendance counts four times per hospital |

**Limitation:** the analysis covers triage categories T2–T5 only. T1 patients which the most critically ill cannot be assessed with this measure.

## 4. Data Exploration

Three grouped summaries, one for each sub-question: how timeliness changed over time, how it differs across districts, and whether hospital workload explains the difference.

All summaries use the NSW, LHD or hospital level separately. Levels are never mixed, since the higher levels aggregate the same patients as the lower ones.

In [ ]:
section("OVERALL DISTRIBUTION")

# Descriptive statistics at hospital level, split by urgency. Hospital level is
# used because it is the finest grain available and shows the true spread.
hosp = triage_clean[triage_clean["reporting_level"] == "Hospital"]
display(
    hosp.groupby("triage_category")["pct_on_time"]
        .describe()[["count", "mean", "std", "min", "50%", "max"]]
        .round(1)
)

In [ ]:
section("CHANGE OVER TIME - NSW")

# NSW-level results for selected years, to show the trend without printing
# all 65 quarters. First quarter of each year is used for a like-for-like view.
nsw = triage_clean[triage_clean["reporting_level"] == "NSW"]
trend = nsw.pivot_table(index="reporting_period", columns="triage_category",
                        values="pct_on_time")

# Reindex by date so the quarters appear in true chronological order
order = nsw.sort_values("quarter_start")["reporting_period"].unique()
trend = trend.reindex(order)

display(trend.loc[["Jan-Mar 2010", "Jan-Mar 2015", "Jan-Mar 2020",
                   "Jan-Mar 2023", "Jan-Mar 2026"]].round(1))

# Change from the first quarter to the last, per triage category
change = (trend.iloc[-1] - trend.iloc[0]).round(1)
display(change.to_frame("change since 2010"))

In [ ]:
section("DIFFERENCE ACROSS DISTRICTS")

# Latest quarter only, so districts are compared on the same conditions
latest = triage_clean["reporting_period"].iloc[-1]
lhd_latest = triage_clean[(triage_clean["reporting_level"] == "LHD")
                          & (triage_clean["reporting_period"] == latest)]

print(f"Quarter: {latest}\n")
display(
    lhd_latest.pivot_table(index="entity", columns="triage_category", values="pct_on_time")
              .sort_values("T2: Emergency")
              .round(1)
)

# Metropolitan vs regional, averaged across the districts in each group
display(
    lhd_latest.groupby(["district_type", "triage_category"])["pct_on_time"]
              .mean().unstack().round(1)
)

In [ ]:
section("WORKLOAD VS PERFORMANCE")

# Latest quarter at hospital level, grouped by peer group. Peer groups sort
# hospitals by size and role, so this shows whether bigger means slower.
hosp_latest = hospitals_clean[hospitals_clean["reporting_period"] == latest]

display(
    hosp_latest.groupby("peer_group")
               .agg(hospitals=("entity", "count"),
                    median_attendances=("attendances", "median"),
                    mean_on_time=("pct_on_time", "mean"))
               .round(1)
               .sort_values("median_attendances", ascending=False)
)

# A single correlation figure summarises the relationship the scatter will show
r = hosp_latest["attendances"].corr(hosp_latest["pct_on_time"])
print(f"\nCorrelation between attendances and on-time %: {r:.2f}")

### What the summaries show

**Triage Urgency Drives Performance Gaps**

On-time treatment rates decline sharply as patient urgency increases. Across 4,801 hospital-quarters, less critical (T5) patients are seen on time 90.9% of the time with high consistency. In contrast, T2 Emergency cases achieve only a 67.3% on-time rate, accompanied by a much wider standard deviation (16.9 points compared to 6.5 for T5). This indicates highly erratic and delayed care for the sickest patients.

**Systemic Deterioration is Confined to Urgent Cases**

Over the last 16 years (Jan–Mar 2010 to Jan–Mar 2026), T2 on-time performance plummeted by 12.3 points (from 69.5% to 57.2%), while lower-urgency categories (T4 and T5) saw marginal improvements. All triage categories hit their absolute lowest performance simultaneously in Jul–Sep 2024, signaling systemic operational strain rather than localized failures.

**Regional Hospitals Outperform Metropolitan Centers**

Proximity to major urban hubs correlates with slower treatment. In early 2026, regional districts outperformed metro areas by 13.9 points in T2 on-time rates (65.2% vs. 51.3%). This geographical disparity is most visible when comparing high-performing regional areas like Murrumbidgee (83.4%) to heavily strained urban networks like South Eastern Sydney (35.3%).

**Patient Volume Explains the Geographic Disparity**

Higher caseloads directly degrade on-time performance, showing a strong negative correlation (r = -0.59). Principal referral hospitals—primarily located in metropolitan networks—manage a median of 20,098 attendances with only a 60.9% on-time rate. Conversely, smaller District Group 2 hospitals handle a median of 2,808 attendances but achieve an 82.3% on-time rate. Hospital size and workload, rather than pure location, drive the metropolitan performance lag.

**The Central Coast Triage Anomaly**

The Central Coast district entirely defies these state-wide trends. It performs strongly for urgent T2 cases (66.8%) but reports the worst metrics for semi-urgent T3 (43.1%) and T4 (46.9%) cases. Because no other district exhibits this severe inverse performance pattern, it likely reflects a highly localized approach to patient flow and queue management rather than a data error.

## 5. Visualisations

I present three charts using distinct formats tailored to the specific data structure of each sub-question. First, I use a line chart to track on-time performance across 65 quarters, providing the most intuitive visualization for continuous time-series data. Next, I employ a sorted horizontal bar chart to compare 17 named districts within a single quarter, converting an otherwise unordered categorical list into a clear, readable hierarchy. Finally, I use a scatter plot to examine the correlation between workload and performance, ensuring every individual hospital remains visible to demonstrate broader systemic patterns between two continuous variables.

In [ ]:
# Chart 1: NSW trend by triage category
fig, ax = plt.subplots(figsize=(10, 5))

nsw = triage_clean[triage_clean["reporting_level"] == "NSW"]

# Shade the pandemic period, so the reader can see the low point falls after it
covid_start = pd.Timestamp("2020-01-01")
covid_end = pd.Timestamp("2022-12-31")
ax.axvspan(covid_start, covid_end, color="#000000", alpha=0.04, zorder=0)

# Labels sit just above the x-axis, where no line ever reaches
ax.text(covid_start + (covid_end - covid_start) / 2, 41.5, "COVID-19",
        fontsize=8, color="#666666", ha="center", va="bottom")

# Mark where NSW Health replaced its access targets
ax.axvline(pd.Timestamp("2025-01-01"), color="#999999", linestyle="--", linewidth=1)
ax.text(pd.Timestamp("2025-01-01"), 41.5, "New targets",
        fontsize=8, color="#666666", ha="center", va="bottom")

colours = ["#c1583f", "#d99058", "#7a9e9f", "#4a7c8c"]
for colour, category in zip(colours, sorted(nsw["triage_category"].unique())):
    data = nsw[nsw["triage_category"] == category].sort_values("quarter_start")
    ax.plot(data["quarter_start"], data["pct_on_time"],
            color=colour, linewidth=1.8, label=category)

ax.set_title("Patients starting treatment on time, NSW emergency departments")
ax.set_ylabel("% on time")
ax.set_ylim(40, 100)
ax.grid(axis="y", alpha=0.3)

# Legend box placed outside the plot area, so it never covers the data
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5),
          frameon=True, edgecolor="#cccccc", fontsize=9)

plt.tight_layout()
plt.savefig(Path.cwd() / "chart1_trend.png", bbox_inches="tight")
plt.show()

The line chart encodes both variables through position on a common scale in the most accurate visual encoding method, while reserving color to distinguish the four triage categories. Displaying four lines approaches the practical limit before a chart becomes visually tangled, which justifies restricting the analysis to T2–T5. Notably, the y-axis originates at 40 rather than zero. While this design choice magnifies the visual slope of the decline, it is defensible: no category approaches zero, and a full axis would waste vertical space. However, it risks overstating the drop for readers who do not verify the scale.

Two annotations provide context that the raw data cannot. Shading the pandemic period clarifies that the lowest performance occurred after the pandemic rather than during it. Additionally, a marker at January 2025 indicates when NSW Health revised its access targets, contextualizing the modest recovery at the chart's right edge. The shading does carry a risk: highlighting a time period invites the audience to infer causality from mere temporal coincidence, a claim explicitly avoided in the accompanying text. A remaining weakness of the chart is that the lines converge tightly between 2010 and 2015, making early disparities difficult to discern.

In [ ]:
# Chart 2: T2 Emergency by district, latest quarter
# T2 is shown alone because it is where the decline concentrated. Sorting by
# value turns 17 unordered districts into a readable ranking.
fig, ax = plt.subplots(figsize=(8, 5.5))

t2 = triage_clean[(triage_clean["reporting_level"] == "LHD")
                  & (triage_clean["reporting_period"] == latest)
                  & (triage_clean["triage_category"] == "T2: Emergency")].sort_values("pct_on_time")

# Colour carries the metropolitan/regional split, so two variables show at once
palette = {"Metropolitan": "#c1583f", "Regional": "#4a7c8c", "Specialty network": "#9a9a9a"}
colours = t2["district_type"].map(palette)

ax.barh(t2["entity"], t2["pct_on_time"], color=colours)
ax.set_title(f"T2 Emergency patients treated on time by district, {latest}")
ax.set_xlabel("% on time")
ax.set_xlim(0, 100)
ax.grid(axis="x", alpha=0.3)

# Build the legend by hand, since colour comes from a mapped column
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in palette.values()]
ax.legend(handles, palette.keys(), frameon=False, loc="lower right")

plt.tight_layout()
plt.savefig(Path.cwd() / "chart2_districts.png", bbox_inches="tight")
plt.show()

The horizontal bar chart succeeds because sorting the bars by value transforms an unordered list of 17 districts into an instantly readable hierarchy. The horizontal orientation ensures that lengthy district names, such as Nepean Blue Mountains, remain legible without relying on awkward, rotated text. Additionally, using color to encode a second variable in the metropolitan versus regional split which allows a single figure to answer two questions simultaneously.

However, this reliance on color is also the chart's primary limitation. Because color is the sole indicator of the geographic split, this distinction is lost in greyscale printing or for readers with color vision deficiencies. While grouping the bars by district type would resolve this accessibility issue, it would fracture the unified ranking that makes the chart so effective. Ultimately, this trade-off was resolved in favor of preserving the continuous ranking.

In [ ]:
# Chart 3: workload against performance, by hospital 
fig, ax = plt.subplots(figsize=(9, 5.5))

hosp_latest = hospitals_clean[hospitals_clean["reporting_period"] == latest]

# Peer groups are NSW Health's own size-and-role classification, ordered here
# from largest to smallest so the colour scale carries that order
peer_order = ["Principal referral", "Major", "Ungrouped acute - tertiary referral",
              "Paediatric specialist", "District group 1", "District group 2"]
peer_colours = ["#8c2f20", "#c1583f", "#d99058", "#b8b8b8", "#7a9e9f", "#4a7c8c"]

for peer, colour in zip(peer_order, peer_colours):
    group = hosp_latest[hosp_latest["peer_group"] == peer]
    ax.scatter(group["attendances"], group["pct_on_time"], label=peer,
               color=colour, s=45, alpha=0.85, edgecolor="white", linewidth=0.6)

# Trend line, so the direction of the relationship is visible rather than implied
slope, intercept = np.polyfit(hosp_latest["attendances"], hosp_latest["pct_on_time"], 1)
x_line = np.array([hosp_latest["attendances"].min(), hosp_latest["attendances"].max()])
ax.plot(x_line, slope * x_line + intercept, color="#666666",
        linestyle="--", linewidth=1, zorder=0)

# Name the two hospitals that break the pattern in opposite directions
for name in ["Westmead", "Liverpool"]:
    point = hosp_latest[hosp_latest["entity"] == name].iloc[0]
    ax.annotate(name, (point["attendances"], point["pct_on_time"]),
                xytext=(6, 6), textcoords="offset points", fontsize=8, color="#444444")

r = hosp_latest["attendances"].corr(hosp_latest["pct_on_time"])
ax.set_title(f"Hospital workload against timeliness, {latest}   (r = {r:.2f}, n = {len(hosp_latest)})")
ax.set_xlabel("Attendances in the quarter")
ax.set_ylabel("% starting treatment on time")
ax.grid(alpha=0.3)
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5),
          frameon=True, edgecolor="#cccccc", fontsize=8)

plt.tight_layout()
plt.savefig(Path.cwd() / "chart3_workload.png", bbox_inches="tight")
plt.show()

A scatter plot is the optimal choice for comparing two continuous variables, as it displays all 73 individual hospitals rather than relying on aggregated summaries. Because the attendance distribution is relatively symmetric (skew = 0.66), a linear axis was retained; applying a logarithmic scale would have compressed the data without enhancing legibility. Color is used to encode peer groups rather than the metropolitan/regional split. Since peer groups scale cleanly with hospital size, this encoding shifts the chart from merely demonstrating that a correlation exists to suggesting what actually drives it. The visualization transparently reflects its statistical limits: a correlation of $r = -0.59$ leaves approximately two-thirds of the variance unexplained. This spread is clearly visible in cases like Westmead and Liverpool, which handle comparable patient volumes yet differ by 37 points in performance. Explicitly labeling these two hospitals is a subjective design choice intended to steer the reader's attention toward this discrepancy.

## 6. Insights and implications

Bringing the three charts together to answer the question directly, and setting out what the answer means for the people affected.

### Patterns and trends

Timeliness in NSW emergency departments has declined primarily for urgent cases. From 2010 to 2026, T5 on-time rates held steady near 90%, while T2 dropped 12.3 points to 57.2%. Consequently, the system is increasingly missing targets for its most time-critical patients.

Geographically, regional districts significantly outperform metropolitan areas in T2 on-time rates (65.2% vs. 51.3% in Q1 2026). This is starkly visible when comparing Murrumbidgee (83.4%) with South Eastern Sydney (35.3%).

Patient volume, rather than location, drives this disparity. Across 73 hospitals, attendances negatively correlate with performance ($r = -0.59$). Urban principal referral hospitals handle massive caseloads (median 20,098) with only 60.9% on-time rates, whereas smaller regional hospitals (median 2,808) achieve 82.3%.

### Unexpected findings

Two notable anomalies diverge from the general trends. First, the Central Coast district performs strongly for urgent T2 cases (66.8%) but ranks lowest state-wide for T3 (43.1%) and T4 (46.9%). This unique reversal across triage categories suggests a highly localized approach to patient flow management rather than a systemic trend. Second, T5 on-time performance briefly spiked to 98.4% in early 2020. This was almost certainly driven by pandemic lockdowns suppressing hospital attendances, effectively acting as a natural experiment that confirms the inverse relationship between patient volume and timeliness observed in the scatter plot.

### Implications

For patients, the most concerning takeaway is that clinical urgency no longer reliably dictates treatment speed; a T2 emergency patient at a large metropolitan hospital faces a lower probability of timely care than a T5 patient nearly anywhere else.

For NSW Health, the strong correlation with workload indicates that facility capacity, rather than clinical process, is the primary binding constraint—though caseload explains only about a third of the overall variance. Notably, Westmead and Liverpool handle comparable patient volumes yet diverge by 37 percentage points in performance, proving that scale is not destiny; investigating the gap between similar hospitals offers the clearest practical opportunity for operational improvement.

Finally, the wide internal spread within New South Wales highlights the danger of relying on aggregated state-level metrics. A single state average would position NSW mid-table, completely concealing a massive 48-point performance range across its internal districts.